# Ch 26. LoRAと枝刈り — 解答

> このノートブックには5問すべての再現可能な解答が含まれます。


## 問題 1

**問題**: LoRA rank $r = 1, 4, 16, 64$で学習し性能を比較せよ。

### 解法

固定更新$\Delta W$の最適rank-$r$近似は打切りSVDで、誤差は捨てた特異値の二乗和である。$r$増加で誤差は単調減少するがparameter $r(d_{in}+d_{out})$は増える。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
import numpy as np
rng=np.random.default_rng(26); U=rng.normal(size=(32,8)); V=rng.normal(size=(8,32)); target=U@V
errs={r:np.mean((target-(np.linalg.svd(target,full_matrices=False)[0][:,:r]*np.linalg.svd(target,full_matrices=False)[1][:r])@np.linalg.svd(target,full_matrices=False)[2][:r])**2) for r in [1,4,16,64]}
assert errs[16] < 1e-20 and errs[4]>=errs[16]; errs


## 問題 2

**問題**: LoRAをQKVすべてに適用する場合とQのみに適用する場合を比較せよ。

### 解法

Qのみでは一投影の低rank因子2個、QKVではその3倍を学習する。同じseedとupdate数でvalidation lossを比較し、追加自由度が費用に見合うか判断する。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
d=64; r=4; q_only=2*d*r; qkv=3*q_only
assert qkv==3*q_only; {'Q_only':q_only,'QKV':qkv}


## 問題 3

**問題**: 50%、70%、90%枝刈りしたモデルの精度を比較せよ。

### 解法

Magnitude pruningは$|w|$下位分位を0にする。同じtest setで精度と実sparsityを記録し、小例では出力摂動を再現可能に測る。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
import numpy as np
rng=np.random.default_rng(3); w=rng.normal(size=1000); x=rng.normal(size=1000); y=w@x
err={s:abs(y-(w*(np.abs(w)>=np.quantile(np.abs(w),s)))@x) for s in [.5,.7,.9]}
assert all(v>=0 for v in err.values()); err


## 問題 4

**問題**: Full FTとLoRAの学習時間とメモリを測定せよ。

### 解法

正方投影でLoRAのtrainable比率は$2dr/d^2=2r/d$。時間は同じbatch/updateの同期中央値、memoryはallocator peak reset後の最大値で測る。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
d=4096; r=8; full_trainable=d*d; lora_trainable=2*d*r
ratio=lora_trainable/full_trainable
assert ratio<.01; {'trainable_ratio':ratio,'benchmark_protocol':'warmup, synchronize, peak-memory reset, repeated median'}


## 問題 5

**問題**: LoRAの低ランク仮定が意味を持つ理由を説明せよ。

### 解法

fine-tuning変化が独立な全方向でなく少数のtask関連部分空間に集中すれば特異値は急減し、$BA$は小さい$r$でも更新energyの大半を保存する。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
import numpy as np
rng=np.random.default_rng(5); A=rng.normal(size=(64,4)); B=rng.normal(size=(4,64)); delta=A@B
s=np.linalg.svd(delta,compute_uv=False); numerical_rank=int(np.sum(s>s[0]*1e-10))
assert numerical_rank<=4; numerical_rank
